# 11.11 · Evaluating and shipping deep agents

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 2 h |
| 📋 Prerequisites | Lessons 11.05–11.10; Lessons 10.03–10.05 and 10.08 (datasets, evaluators, trajectory evals, CI gates). |
| 🔑 Keys | none for 🧪 cells · optional 💸 cell ≈ 10–20K tokens (off by default) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Evaluate** a deep agent on four axes: trajectory, file state, grounding, and cost.
2. **Measure** reliability over repeated runs (pass rate with an interval, pass^k, flip rate).
3. **Gate** a release on quality, reliability and cost, and prove the gate blocks a bad build.
4. **Package** an agent for deployment (`langgraph.json`) and harden the upgrade path.

## 📚 Official reading (verified 2026-10-16, `deepagents` 0.7.26)

| Page | Read it for |
|---|---|
| [Going to production](https://docs.langchain.com/oss/python/deepagents/going-to-production) | Deployments, `langgraph.json`, durability, multi-tenancy, guardrails |
| [Trajectory evaluations](https://docs.langchain.com/langsmith/trajectory-evals) | Match modes and LLM-as-judge on trajectories |
| [Application structure](https://docs.langchain.com/langsmith/application-structure) | `langgraph.json` fields |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8" "langchain-google-genai>=4.4.1,<5" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-eval-ship'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

---
## Part 1 · What to check in a deep-agent run

| Axis | Question | `mastery.deep` check |
|---|---|---|
| Trajectory | Did it search, get the draft criticised, then write? Nothing forbidden? | `trajectory_matches(..., mode="ordered")`, `forbid_tools` |
| File state | Is the deliverable there, with the required sections? | `file_text`, `file_contains` |
| Grounding | Is every cited URL one a tool actually returned? | `grounding()` |
| Cost | Within the call budget? | `max_model_calls` (`run_stats`) |

All four are deterministic and free. Keep LLM judges (Lesson 10.04) for what code can't check, such as "is the brief *useful*?", and validate them first.

In [ ]:
# 🧪 A tiny research agent we can evaluate for free: a fake web search, a critic subagent, a report file.
import random

from langchain.tools import tool
from langchain_core.messages import AIMessage

from mastery.deep import build_deep_agent, check_case, run_stats, trajectory
from mastery.testing import ScriptedChatModel

SOURCES = {
    "solar": "https://example.org/solar-2026",
    "heat pumps": "https://example.org/heat-pumps",
    "ev charging": "https://example.org/ev-charging",
    "batteries": "https://example.org/home-batteries",
}


@tool
def web_search(query: str) -> str:
    """Search the web (offline fixture: one fixed page per topic)."""
    topic = next((t for t in SOURCES if t in query.lower()), None)
    return f"Findings for {query!r}.\nSources:\n- {SOURCES[topic]}" if topic else "No results.\nSources:\n- (none)"


def calls(*items) -> AIMessage:
    return AIMessage("", tool_calls=[{"name": n, "args": a, "id": i} for n, a, i in items])


def research_script(topic: str, *, skip_critic: bool, invent: bool) -> list:
    url = SOURCES[topic]
    cited = url + (" and https://made-up.example/stats" if invent else "")
    report = f"# Brief: {topic}\n\n## Findings\nCosts fell in 2026 ({cited}).\n\n## Sources\n- {url}\n"
    steps = [calls(("web_search", {"query": f"{topic} costs 2026"}, "s"))]
    if not skip_critic:
        steps.append(calls(("task", {"subagent_type": "critic", "description": "Check the draft claims."}, "c")))
    steps += [calls(("write_file", {"file_path": "/reports/brief.md", "content": report}, "w")), AIMessage("Brief saved.")]
    return steps


def make_agent(topic: str, variant: str, run: int):
    rng = random.Random(f"{variant}-{topic}-{run}")            # deterministic "flakiness" per (variant, case, run)
    flaky = variant == "v1"
    script = research_script(topic, skip_critic=flaky and rng.random() < 0.35, invent=flaky and rng.random() < 0.25)
    critic = {"name": "critic", "description": "Checks a draft's claims against its sources.",
              "system_prompt": "You check claims.", "model": ScriptedChatModel(responses=[AIMessage("Claims check out.")])}
    return build_deep_agent(ScriptedChatModel(responses=script), tools=[web_search], subagents=[critic], fallbacks=False)


CASES = [{"id": f"q{i}", "topic": t, "input": f"Write a short brief on {t} costs in 2026.",
          "expect_tools": ["web_search", "task", "write_file"], "forbid_tools": ["execute"],
          "file": "/reports/brief.md", "file_contains": ["## Findings", "## Sources"], "grounded": True,
          "max_model_calls": 6} for i, t in enumerate(SOURCES, 1)]


def run_case(case, variant, run):
    agent = make_agent(case["topic"], variant, run)
    result = agent.invoke({"messages": [{"role": "user", "content": case["input"]}]})
    return result, check_case(result, case)


result, checks = run_case(CASES[0], "v2", 0)
print("trajectory:", trajectory(result))
print("checks    :", checks)
print("stats     :", run_stats(result))

**You should now see** the trajectory `['web_search', 'task', 'write_file']`, every check `True`, and 4 model calls. `ordered` mode allows extra calls in between (re-searching is fine); `exact` would break on harmless variation.

## Part 2 · Repeated runs

In [ ]:
# 🧪 k = 3 runs × 4 cases per variant: pass rate with an interval, pass^k, and which check fails.
from collections import Counter

from mastery.evals.stats import summarize_repeats

K = 3


def evaluate(variant):
    runs, failures = [], Counter()
    for run in range(K):
        scores = {}
        for case in CASES:
            _, checks = run_case(case, variant, run)
            scores[case["id"]] = checks["passed"]
            failures.update(name for name, ok in checks.items() if not ok and name != "passed")
        runs.append(scores)
    return summarize_repeats(runs), failures


summaries = {}
for variant in ("v1", "v2"):
    summary, failures = evaluate(variant)
    summaries[variant] = summary
    print(f"{variant}: pass rate {summary.mean:.0%} | pass^{K} {summary.pass_hat_k:.0%} | flip rate {summary.flip_rate:.0%} "
          f"| failing checks {dict(failures)}")

**You should now see** v1 at about 42% with pass^3 of 0% and a 100% flip rate: every case fails at least once, mostly by skipping the critic (trajectory) or citing an invented URL (grounded). v2 passes everything. One run of v1 could easily have looked fine: the interval and pass^k are what reveal it.

> ⚠️ **Production note: four cases is a smoke test, not an eval.** The interval is wide and only as good as the cases. In production: 30+ cases drawn from real requests (Lesson 10.03), k ≥ 3 runs, a held-out set, and a failure-category breakdown, like the `failing checks` counter above, so you fix causes and not averages.

## Part 3 · The release gate

In [ ]:
# 🧪 A release gate for the agent: quality AND reliability AND cost. v1 must be blocked, v2 shipped.
from mastery.evals.gates import GateReport, GateResult, cost_gate, summary_gate


def gate(variant):
    s = summaries[variant]
    calls_per_run = run_stats(run_case(CASES[0], variant, 0)[0])["model_calls"]
    return GateReport().add(
        summary_gate("pass rate", s, min_lower_bound=0.5),
        GateResult("pass^3", s.pass_hat_k >= 0.75, f"{s.pass_hat_k:.0%} vs ≥ 75%"),
        cost_gate("model calls", calls_per_run, 6, unit=" calls"))


for variant in ("v1", "v2"):
    report = gate(variant)
    print(f"--- {variant}\n{report.to_markdown()}\n")

**You should now see** v1 blocked by `pass rate` and `pass^3`, and v2 with all gates passed. A gate you've never seen fail is untested: always keep a known-bad build (here v1) in CI.

## Part 4 · Shipping

In [ ]:
# 🧪 Shipping files: langgraph.json (LangSmith Deployment / `langgraph dev`) + the agent module it points to.
import json
import tempfile
from pathlib import Path

app = Path(tempfile.mkdtemp(prefix="m11-ship-"))
(app / "agent.py").write_text(
    "from mastery.deep import build_deep_agent\n\n"
    "# A module-level graph: the platform imports it and provides the checkpointer and store.\n"
    "agent = build_deep_agent(system_prompt='You are a research assistant.')\n", encoding="utf-8")
(app / "langgraph.json").write_text(json.dumps(
    {"dependencies": ["."], "graphs": {"agent": "./agent.py:agent"}, "env": ".env"}, indent=2), encoding="utf-8")
print((app / "langgraph.json").read_text(encoding="utf-8"))
print(sorted(p.name for p in app.iterdir()))

**What just happened?** `langgraph.json` maps a graph ID (`agent`) to a module-level variable. `langgraph dev` runs it locally, and a LangSmith Deployment builds it, adding the checkpointer, store, threads and auth for you. The docs recommend **Managed Deep Agents** (public beta) as the default path, and a LangSmith Deployment when you need custom code or auth (Lesson 11.12, F6).

**The upgrade path** is where deep agents break (pre-1.0):

1. Pin the exact version in a lock file (`deepagents==0.7.26`).
2. Turn deprecations into failures in CI: `python -W error::DeprecationWarning -m pytest`.
3. Run this lesson's suite (k = 3) as the gate on every bump; compare the failing-check counts, not just the pass rate.
4. Keep beta features behind your own wrapper (`mastery/deep/`), so a change touches one file.

## Part 5 · 💸 One live case

> 💸 Calls your deep model (`MASTERY_DEEP_MODEL`, default Gemini free tier). Rate-limited to one call every 4 s and capped per run. Without the key it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE (opt-in): one case on your real model, scored by the same checks. ≈ 10–20K tokens.
RUN_LIVE = False
if RUN_LIVE and HAVE_DEEP_KEY:
    from mastery.deep import deep_model
    critic = {"name": "critic", "description": "Checks a draft's claims against its sources.",
              "system_prompt": "Check each claim against the cited sources. Reply with problems only.",
              "model": deep_model("subagent")}
    live = build_deep_agent(tools=[web_search], subagents=[critic], model_calls=8, tool_calls=10,
                            system_prompt=("Research with web_search, ask the critic to check your draft, then write "
                                           "/reports/brief.md with '## Findings' and '## Sources'. Cite only URLs you found."))
    case = CASES[0]
    result = live.invoke({"messages": [{"role": "user", "content": case["input"]}]})
    print("trajectory:", trajectory(result))
    print("checks    :", check_case(result, case))
    print("stats     :", run_stats(result))
elif not RUN_LIVE:
    print("⏭️ Skipped: set RUN_LIVE = True to try it (≈ 10–20K tokens).")
else:
    print(f"⏭️ Skipped: no key for {DEEP_MODEL} (see the setup cell).")

**You should now see** (when enabled) a real trajectory and the same checks. Real models fail differently from scripts: look at **which** check fails before changing the prompt. On the free tier, run more cases on different days rather than all at once.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Trajectory check fails on a good run | `exact` mode | `ordered` with only the essential steps |
| Subagent calls missing from `trajectory()` | They live inside the `task` call | Evaluate subagents separately, or use `subagents_called()` |
| "Grounded" passes with no citations | No citation requirement | `check_case` requires at least one cited URL |
| Gate green on a broken build | Gate never tested on a bad build | Keep a known-bad variant in CI |
| Deployment can't import the graph | Path in `graphs` wrong, or the agent is built inside `if __name__ == "__main__"` | `"./agent.py:agent"`, module-level variable |

## 🏋️ Exercises

1. **(Easy)** Switch the trajectory mode to `exact` and rerun Part 2. What changes for v2, and why is it still fine here?
2. **(Medium)** Add a fifth case whose topic has no source. What should the agent do, and which check would you add?
3. **(Medium)** Raise the pass-rate bar to a lower bound ≥ 0.9. Does v2 still ship with n = 12? Why?
4. **(Stretch)** Add a cost gate on tokens (`input_tokens + output_tokens` from `run_stats`) at 20% above v2's mean.

<details>
<summary><b>Show solutions</b></summary>

**1.** Nothing changes for v2, whose script is always exactly `web_search → task → write_file`. Real models re-search or re-read, which `exact` would fail.

**2.** It should say there is no reliable source and **not** write findings. Add `"file_contains": ["no reliable source"]`, or a check that the brief cites nothing invented: `grounding()` with zero invented and a refusal phrase.

**3.** Yes: 12/12 gives a cluster-bootstrap interval of [100%, 100%] here. With a real model a single failure in 12 would pull the Wilson lower bound to about 65% (`wilson_interval(11, 12)`), which is why bars need enough cases.

**4.** `cost_gate("tokens", mean_tokens_v1, 1.2 * mean_tokens_v2, unit=" tok")`, with the means computed from `run_stats` over the k runs.
</details>

## 🔎 Docs drill (5 min)

In langgraph.json, what does the value of a graphs entry point to, and which field loads your API keys?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Application structure](https://docs.langchain.com/langsmith/application-structure) or the `langgraph.json` table in [Going to production](https://docs.langchain.com/oss/python/deepagents/going-to-production).
2. `"<id>": "./<file>:<variable>"`: the compiled graph (or a constructor function) exported from that file.
3. `env`: a path to a `.env` file (or inline variables in JS). Never commit it.

</details>

## ✅ Checkpoint

- [ ] I can evaluate a run on trajectory, file state, grounding and cost.
- [ ] I can report pass rate with an interval, pass^k and flip rate, and say why one run isn't enough.
- [ ] I can write a release gate and prove it blocks a bad build.
- [ ] I can package an agent with `langgraph.json` and describe a safe upgrade path.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

Check the current status of Managed Deep Agents (beta or GA?) and the `langgraph` CLI version on [Going to production](https://docs.langchain.com/oss/python/deepagents/going-to-production). Log both with the date in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Evaluate deep agents on what they **did** (trajectory, files, sources, cost), not only on what they said.
- Repeat runs; gate on interval lower bounds and pass^k; keep a known-bad build in CI.
- Ship with `langgraph.json`; pin, fail on deprecations, and re-run the suite on every upgrade.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 11-12-frontier-briefings.ipynb** → [`11-12-frontier-briefings.ipynb`](11-12-frontier-briefings.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)